# TEA-GNN on TKGL-Wikidata Cross-Lingual (sampled, memory-safe)

**TEA-GNN** (Xu, Su, Lehmann — EMNLP 2021) on `tkgl_wikidata_crosslingual`.

Full TKGL can have **~1.2M EN entities / ~9M quads** → OOM on Colab.
This notebook **samples** an alignment-centric subgraph so TEA-GNN fits limited GPU RAM.

### Sampling strategy
1. Load all alignment pairs, then keep `TRAIN_SEED + DEV_SIZE` pairs
2. Keep only entities that appear in those pairs (+ optional 1-hop neighbors)
3. Keep only quadruples whose both endpoints are in the kept entity set
4. Optional hard caps: `MAX_QUADS_EN`, `MAX_QUADS_OTHER`

### Zip layout
```
tkgl_wikidata_crosslingual.zip
├── en_fa/  en_zh/  en_ja/  en_fr/
    ├── alignment_all.tsv
    ├── quadruples_en.tsv      # h r t ts
    └── quadruples_<lang>.tsv
```

### Colab
1. **Runtime → Restart session** (required after any prior OOM)
2. GPU runtime (T4 is enough with defaults)
3. Put zip on Drive or `/content`, set `LANG_PAIR`, run top→bottom

**Defaults fit T4 (~15GB):** `DIM=64`, `DEPTH=1`, `TRAIN_SEED=2000`, `DEV_SIZE=3000`, `HOP_EXPANSION=1`.


In [1]:
import os, gc, zipfile, random
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.empty_cache()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GB:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))


Device: cpu


In [2]:
# Optional: mount Drive if zip is there
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Drive mount skipped:", e)


Mounted at /content/drive


In [3]:
# ============================================================
# Config — tune these if you still OOM
# ============================================================
from pathlib import Path

LANG_PAIR = "en_ja"  # en_fa | en_zh | en_ja | en_fr

ZIP_CANDIDATES = [
    Path("/content/drive/MyDrive/tkgl_wikidata_crosslingual.zip"),
    Path("/content/tkgl_wikidata_crosslingual.zip"),
]

# ---- sampling (main anti-OOM knobs) ----
TRAIN_SEED = 2000       # # seed alignment pairs for training
DEV_SIZE   = 3000       # # pairs held out for eval
HOP_EXPANSION = 1       # MUST be >=1 on TKGL; 0 leaves almost no edges
MAX_QUADS_EN = 800_000  # hard cap while streaming EN quads (None = no cap)
MAX_QUADS_OTHER = 400_000

# ---- model / train ----
DIM = 64
DEPTH = 1               # TEA-GNN layers
DROPOUT = 0.3
LR = 1e-3
EPOCHS = 120
MARGIN = 1.0
NEG_SIZE = 32           # negatives per positive (sampled)
BATCH_PAIRS = 512
EVAL_EVERY = 10
TOP_K = (1, 5, 10)
CSLS_K = 10
EVAL_MAX_PAIRS = 3000   # None = all dev pairs

# ---- memory ----
USE_CHECKPOINT = True
CHUNK_EDGES = 65536
print("LANG_PAIR =", LANG_PAIR)
print("TRAIN_SEED / DEV_SIZE / HOP =", TRAIN_SEED, DEV_SIZE, HOP_EXPANSION)
print("DIM / DEPTH =", DIM, DEPTH)


LANG_PAIR = en_ja
TRAIN_SEED / DEV_SIZE / HOP = 2000 3000 1
DIM / DEPTH = 64 1


In [4]:
# ============================================================
# Locate + extract zip (robust to nested layouts)
# ============================================================
from pathlib import Path
import zipfile

if "ZIP_CANDIDATES" not in globals():
    LANG_PAIR = globals().get("LANG_PAIR", "en_ja")
    ZIP_CANDIDATES = [
        Path("/content/drive/MyDrive/tkgl_wikidata_crosslingual.zip"),
        Path("/content/tkgl_wikidata_crosslingual.zip"),
    ]

zip_path = next((p for p in ZIP_CANDIDATES if p.exists()), None)
if zip_path is None:
    found = list(Path("/content").rglob("*crosslingual*.zip")) + list(Path("/content").rglob("en_ja.zip"))
    zip_path = found[0] if found else None
if zip_path is None:
    raise FileNotFoundError(
        "Could not find tkgl_wikidata_crosslingual.zip. "
        "Upload it to /content or Drive and re-run Config."
    )
print("Using zip:", zip_path)

extract_root = Path("/content/tkgl_extract")
extract_root.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    names = zf.namelist()
    members = [n for n in names if f"/{LANG_PAIR}/" in n or n.startswith(LANG_PAIR + "/")]
    if not members:
        print("WARNING: no members matched LANG_PAIR; extracting all (slow).")
        members = names
    print(f"Extracting {len(members)} members...")
    for m in members:
        zf.extract(m, extract_root)

candidates = list(extract_root.rglob("alignment_all.tsv"))
data_dir = None
for al in candidates:
    parent = al.parent
    q_en = parent / "quadruples_en.tsv"
    if q_en.exists() and any(parent.glob("quadruples_*.tsv")):
        # need at least one non-en quadruple file
        others = [p for p in parent.glob("quadruples_*.tsv") if p.name != "quadruples_en.tsv"]
        if others:
            data_dir = parent
            break
if data_dir is None:
    raise FileNotFoundError(f"Could not locate pair folder under {extract_root}")

align_path = data_dir / "alignment_all.tsv"
quad_en = data_dir / "quadruples_en.tsv"
quad_other = next(p for p in sorted(data_dir.glob("quadruples_*.tsv")) if p.name != "quadruples_en.tsv")
print("data_dir:", data_dir)
print("alignment:", align_path)
print("quad_en:", quad_en)
print("quad_other:", quad_other)


Using zip: /content/drive/MyDrive/tkgl_wikidata_crosslingual.zip
Extracting 7 members...
data_dir: /content/tkgl_extract/tkgl_wikidata_crosslingual/en_ja
alignment: /content/tkgl_extract/tkgl_wikidata_crosslingual/en_ja/alignment_all.tsv
quad_en: /content/tkgl_extract/tkgl_wikidata_crosslingual/en_ja/quadruples_en.tsv
quad_other: /content/tkgl_extract/tkgl_wikidata_crosslingual/en_ja/quadruples_ja.tsv


In [5]:
# ============================================================
# Load alignment + sample pairs
# ============================================================

def load_alignment(path):
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        header = f.readline()
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            pairs.append((int(parts[0]), int(parts[1])))
    return pairs

all_pairs = load_alignment(align_path)
print(f"alignment pairs total: {len(all_pairs):,}")

rng = random.Random(SEED)
rng.shuffle(all_pairs)

n_train = min(TRAIN_SEED, max(1, len(all_pairs) // 5))
n_dev = min(DEV_SIZE, max(1, len(all_pairs) - n_train))
train_raw = all_pairs[:n_train]
dev_raw = all_pairs[n_train:n_train + n_dev]
print(f"sampled train={len(train_raw)}  dev={len(dev_raw)}")

keep1 = {a for a, _ in train_raw + dev_raw}
keep2 = {b for _, b in train_raw + dev_raw}
print(f"alignment entities: EN={len(keep1):,}  other={len(keep2):,}")


alignment pairs total: 132,769
sampled train=2000  dev=3000
alignment entities: EN=5,000  other=5,000


In [6]:
# ============================================================
# Stream-load quadruples, filter to kept entities (+ 1-hop neighbors)
# ============================================================
# Critical: with only alignment entities (hop=0), almost NO edges remain
# inside each KG, so the GNN never sees structure and Hits@1 ≈ 0.

def stream_quads(path, max_rows=None):
    n = 0
    with open(path, "r", encoding="utf-8") as f:
        first = f.readline()
        parts = first.rstrip("\n").split("\t")
        try:
            h, r, t, ts = int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
            yield h, r, t, ts
            n += 1
        except Exception:
            pass
        for line in f:
            if max_rows is not None and n >= max_rows:
                break
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 4:
                continue
            try:
                yield int(parts[0]), int(parts[1]), int(parts[2]), int(parts[3])
                n += 1
            except Exception:
                continue


def collect_filtered(path, seed_ents, max_rows=None, expand_hops=1):
    """Keep seed entities + neighbors within expand_hops, and all edges among them."""
    seed_ents = set(seed_ents)
    # Pass A: discover neighbors of seeds
    nbrs = set()
    if expand_hops > 0:
        for h, r, t, ts in stream_quads(path, max_rows=max_rows):
            if h in seed_ents:
                nbrs.add(t)
            if t in seed_ents:
                nbrs.add(h)
    keep_ents = set(seed_ents) | nbrs
    print(f"  seeds={len(seed_ents):,}  +neighbors={len(nbrs):,}  keep={len(keep_ents):,}")

    # Pass B: keep triples fully inside keep_ents
    triples = []
    rels, times = set(), set()
    for h, r, t, ts in stream_quads(path, max_rows=max_rows):
        if h in keep_ents and t in keep_ents:
            triples.append((h, r, t, ts))
            rels.add(r); times.add(ts)
    return triples, keep_ents, rels, times


print("Loading EN quadruples (filtered)...")
seed1 = set(keep1)
triples1, keep1, r1, t1 = collect_filtered(
    quad_en, seed1, max_rows=MAX_QUADS_EN, expand_hops=HOP_EXPANSION
)
print(f"  EN triples={len(triples1):,}  ents={len(keep1):,}  rels={len(r1)}  times={len(t1)}")

print("Loading OTHER quadruples (filtered)...")
seed2 = set(keep2)
triples2, keep2, r2, t2 = collect_filtered(
    quad_other, seed2, max_rows=MAX_QUADS_OTHER, expand_hops=HOP_EXPANSION
)
print(f"  OT triples={len(triples2):,}  ents={len(keep2):,}  rels={len(r2)}  times={len(t2)}")

if len(triples1) < 100 or len(triples2) < 100:
    print("WARNING: very few triples — GNN cannot learn structure. Increase MAX_QUADS_* or HOP_EXPANSION.")

train_raw = [(a, b) for a, b in train_raw if a in keep1 and b in keep2]
dev_raw   = [(a, b) for a, b in dev_raw   if a in keep1 and b in keep2]
print(f"pairs after filter: train={len(train_raw)} dev={len(dev_raw)}")


Loading EN quadruples (filtered)...
  seeds=5,000  +neighbors=521  keep=5,517
  EN triples=35,925  ents=5,517  rels=47  times=1707
Loading OTHER quadruples (filtered)...
  seeds=5,000  +neighbors=521  keep=5,509
  OT triples=40,037  ents=5,509  rels=59  times=1812
pairs after filter: train=2000 dev=3000


In [7]:
# ============================================================
# Remap to contiguous IDs (KG1 then KG2)
# ============================================================

def make_map(ids):
    ids = sorted(ids)
    return {x: i for i, x in enumerate(ids)}

e1_map = make_map(keep1)
e2_map = make_map(keep2)
r_map = make_map(r1 | r2)
t_map = make_map(t1 | t2)

N1 = len(e1_map)
N2 = len(e2_map)
NUM_ENT = N1 + N2
NUM_REL = len(r_map) * 2   # forward + reverse
NUM_TIME = len(t_map)

def remap_triples(triples, emap):
    out = []
    base = len(r_map)
    for h, r, t, ts in triples:
        if h not in emap or t not in emap:
            continue
        hi, ti = emap[h], emap[t]
        ri = r_map[r]
        tsi = t_map[ts]
        out.append((hi, ri, ti, tsi))
        out.append((ti, ri + base, hi, tsi))  # reverse
    return out

e2_map_shift = {k: v + N1 for k, v in e2_map.items()}

triples1_m = remap_triples(triples1, e1_map)
triples2_m = remap_triples(triples2, e2_map_shift)

train_pair = np.array([(e1_map[a], e2_map_shift[b]) for a, b in train_raw], dtype=np.int64)
dev_pair   = np.array([(e1_map[a], e2_map_shift[b]) for a, b in dev_raw], dtype=np.int64)

print(f"NUM_ENT={NUM_ENT} (N1={N1}, N2={N2})  NUM_REL={NUM_REL}  NUM_TIME={NUM_TIME}")
print(f"edges={len(triples1_m)+len(triples2_m):,}  train_pair={len(train_pair)}  dev_pair={len(dev_pair)}")

all_edges = triples1_m + triples2_m
src = torch.tensor([e[0] for e in all_edges], dtype=torch.long)
dst = torch.tensor([e[2] for e in all_edges], dtype=torch.long)
rel = torch.tensor([e[1] for e in all_edges], dtype=torch.long)
tim = torch.tensor([e[3] for e in all_edges], dtype=torch.long)
print("edge tensors ready:", src.shape[0])

# degree diagnostic — if avg degree ~0, model cannot learn
deg = np.zeros(NUM_ENT, dtype=np.int64)
for e in all_edges:
    deg[e[2]] += 1
print(f"avg in-degree={deg.mean():.2f}  median={np.median(deg):.1f}  max={deg.max()}")
print(f"isolated entities: {(deg==0).sum()} / {NUM_ENT}")


NUM_ENT=11026 (N1=5517, N2=5509)  NUM_REL=126  NUM_TIME=1812
edges=151,924  train_pair=2000  dev_pair=3000
edge tensors ready: 151924
avg in-degree=13.78  median=0.0  max=7990
isolated entities: 9457 / 11026


In [8]:
# ============================================================
# TEA-GNN model (time-aware attention, memory-friendly)
# ============================================================

class TEALayer(nn.Module):
    '''Time-aware GNN layer: attention modulated by relation + timestamp embeddings.
    Aggregates neighbour messages in edge chunks to limit peak memory.
    '''
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.dim = dim
        self.dropout = dropout
        self.w = nn.Linear(dim, dim, bias=False)
        self.a = nn.Linear(dim * 2, 1, bias=False)

    def forward(self, x, src, dst, rel_emb, time_emb, rel_idx, time_idx, chunk=65536):
        N, D = x.size()
        E = src.size(0)
        h = self.w(x)
        out = torch.zeros_like(h)
        deg = torch.zeros(N, device=x.device, dtype=x.dtype)

        for start in range(0, E, chunk):
            end = min(start + chunk, E)
            s = src[start:end]
            d = dst[start:end]
            r = rel_emb[rel_idx[start:end]]
            t = time_emb[time_idx[start:end]]
            neigh = h[s] * torch.tanh(r + t)
            cat = torch.cat([h[d], neigh], dim=-1)
            score = self.a(cat).squeeze(-1)
            alpha = torch.sigmoid(score)
            msg = alpha.unsqueeze(-1) * neigh
            out.index_add_(0, d, msg)
            deg.index_add_(0, d, alpha)

        deg = deg.clamp(min=1e-6).unsqueeze(-1)
        out = out / deg
        out = F.dropout(out, p=self.dropout, training=self.training)
        return F.relu(out + h)


class TEAGNN(nn.Module):
    def __init__(self, num_ent, num_rel, num_time, dim, depth, dropout=0.3):
        super().__init__()
        self.ent_emb = nn.Embedding(num_ent, dim)
        self.rel_emb = nn.Embedding(num_rel, dim)
        self.time_emb = nn.Embedding(num_time, dim)
        self.layers = nn.ModuleList([TEALayer(dim, dropout) for _ in range(depth)])
        nn.init.xavier_uniform_(self.ent_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)
        nn.init.xavier_uniform_(self.time_emb.weight)

    def forward(self, src, dst, rel, tim, chunk=65536):
        x = self.ent_emb.weight
        for layer in self.layers:
            if USE_CHECKPOINT and self.training:
                x = torch.utils.checkpoint.checkpoint(
                    layer, x, src, dst, self.rel_emb.weight, self.time_emb.weight,
                    rel, tim, chunk, use_reentrant=False,
                )
            else:
                x = layer(x, src, dst, self.rel_emb.weight, self.time_emb.weight, rel, tim, chunk)
        return F.normalize(x, p=2, dim=1)


model = TEAGNN(NUM_ENT, NUM_REL, NUM_TIME, DIM, DEPTH, DROPOUT).to(DEVICE)
src_d = src.to(DEVICE)
dst_d = dst.to(DEVICE)
rel_d = rel.to(DEVICE)
tim_d = tim.to(DEVICE)
print(model)
print("params:", sum(p.numel() for p in model.parameters()))


TEAGNN(
  (ent_emb): Embedding(11026, 64)
  (rel_emb): Embedding(126, 64)
  (time_emb): Embedding(1812, 64)
  (layers): ModuleList(
    (0): TEALayer(
      (w): Linear(in_features=64, out_features=64, bias=False)
      (a): Linear(in_features=128, out_features=1, bias=False)
    )
  )
)
params: 833920


In [9]:
# ============================================================
# Loss + metrics
# ============================================================

def margin_loss(emb, pairs, neg_size, margin, n1, n2):
    if len(pairs) == 0:
        return emb.sum() * 0.0
    idx = np.random.choice(len(pairs), size=min(BATCH_PAIRS, len(pairs)), replace=False)
    pos = pairs[idx]
    left = torch.from_numpy(pos[:, 0]).to(DEVICE)
    right = torch.from_numpy(pos[:, 1]).to(DEVICE)
    pos_sc = (emb[left] * emb[right]).sum(dim=1)

    neg_r = torch.randint(n1, n1 + n2, (left.size(0), neg_size), device=DEVICE)
    neg_l = torch.randint(0, n1, (right.size(0), neg_size), device=DEVICE)

    left_exp = emb[left].unsqueeze(1)
    neg_sc_r = (left_exp * emb[neg_r]).sum(dim=-1)
    right_exp = emb[right].unsqueeze(1)
    neg_sc_l = (right_exp * emb[neg_l]).sum(dim=-1)

    loss_r = F.relu(margin - pos_sc.unsqueeze(1) + neg_sc_r).mean()
    loss_l = F.relu(margin - pos_sc.unsqueeze(1) + neg_sc_l).mean()
    return loss_r + loss_l


def csls_hits(emb_np, pairs, top_k=(1, 5, 10), csls_k=10, max_pairs=None):
    if len(pairs) == 0:
        print("no pairs"); return
    pairs = pairs[:max_pairs] if max_pairs else pairs
    L = emb_np[pairs[:, 0]]
    R_all = emb_np[N1:N1 + N2]
    sim = L @ R_all.T
    k = min(csls_k, sim.shape[1])
    row_part = np.partition(sim, -k, axis=1)[:, -k:].mean(axis=1, keepdims=True)
    k2 = min(csls_k, sim.shape[0])
    col_part = np.partition(sim, -k2, axis=0)[-k2:, :].mean(axis=0, keepdims=True)
    sim = 2 * sim - row_part - col_part

    targets = pairs[:, 1] - N1
    ranks = []
    for i in range(len(pairs)):
        order = np.argsort(-sim[i])
        rank = int(np.where(order == targets[i])[0][0]) + 1
        ranks.append(rank)
    ranks = np.array(ranks)
    hits = {k: float((ranks <= k).mean()) for k in top_k}
    mrr = float((1.0 / ranks).mean())
    msg = "  ".join([f"Hits@{k}: {hits[k]*100:.2f}%" for k in top_k])
    print(f"  {msg}  MRR: {mrr:.4f}  (n={len(ranks)})")
    return hits, mrr


In [10]:
# ============================================================
# Train
# ============================================================

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print("Training...")
print(f"edges={src_d.size(0):,}  train_pair={len(train_pair)}  dev_pair={len(dev_pair)}")
for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    emb = model(src_d, dst_d, rel_d, tim_d, chunk=CHUNK_EDGES)
    loss = margin_loss(emb, train_pair, NEG_SIZE, MARGIN, N1, N2)
    loss.backward()
    optimizer.step()

    if epoch == 0 and torch.cuda.is_available():
        print(f"[epoch 001] peak GPU mem: "
              f"{torch.cuda.max_memory_allocated()/2**30:.2f} GiB / "
              f"{torch.cuda.get_device_properties(0).total_memory/2**30:.2f} GiB")

    if (epoch + 1) % EVAL_EVERY == 0 or epoch == EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            emb_np = model(src_d, dst_d, rel_d, tim_d, chunk=CHUNK_EDGES).detach().cpu().numpy()
        print(f"[epoch {epoch+1:03d}] loss={loss.item():.4f}")
        print("  train:", end=" ")
        csls_hits(emb_np, train_pair, TOP_K, CSLS_K, min(1000, len(train_pair)))
        print("  dev:  ", end=" ")
        csls_hits(emb_np, dev_pair, TOP_K, CSLS_K, EVAL_MAX_PAIRS)
        del emb_np
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    del emb, loss
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Training complete.")


Training...
edges=151,924  train_pair=2000  dev_pair=3000
[epoch 010] loss=1.2456
  train:   Hits@1: 44.00%  Hits@5: 56.00%  Hits@10: 60.20%  MRR: 0.4945  (n=1000)
  dev:     Hits@1: 0.00%  Hits@5: 0.07%  Hits@10: 0.13%  MRR: 0.0013  (n=3000)
[epoch 020] loss=0.6689
  train:   Hits@1: 84.50%  Hits@5: 89.00%  Hits@10: 90.50%  MRR: 0.8663  (n=1000)
  dev:     Hits@1: 0.00%  Hits@5: 0.10%  Hits@10: 0.10%  MRR: 0.0013  (n=3000)
[epoch 030] loss=0.4123
  train:   Hits@1: 96.70%  Hits@5: 98.20%  Hits@10: 98.70%  MRR: 0.9745  (n=1000)
  dev:     Hits@1: 0.03%  Hits@5: 0.10%  Hits@10: 0.17%  MRR: 0.0016  (n=3000)
[epoch 040] loss=0.3019
  train:   Hits@1: 98.90%  Hits@5: 99.40%  Hits@10: 99.60%  MRR: 0.9918  (n=1000)
  dev:     Hits@1: 0.00%  Hits@5: 0.07%  Hits@10: 0.13%  MRR: 0.0012  (n=3000)
[epoch 050] loss=0.2336
  train:   Hits@1: 99.30%  Hits@5: 99.90%  Hits@10: 99.90%  MRR: 0.9956  (n=1000)
  dev:     Hits@1: 0.00%  Hits@5: 0.10%  Hits@10: 0.27%  MRR: 0.0014  (n=3000)
[epoch 060] loss=

In [11]:
# ============================================================
# Final eval + save
# ============================================================
model.eval()
with torch.no_grad():
    final = model(src_d, dst_d, rel_d, tim_d, chunk=CHUNK_EDGES).detach().cpu().numpy()

print("embedding shape", final.shape)
print("Final CSLS on dev:")
csls_hits(final, dev_pair, TOP_K, CSLS_K, max_pairs=None)

save_path = f"/content/drive/MyDrive/tea_gnn_tkgl_{LANG_PAIR}_sampled.pt"
try:
    torch.save({
        "model": model.state_dict(),
        "config": {
            "LANG_PAIR": LANG_PAIR,
            "TRAIN_SEED": TRAIN_SEED,
            "DEV_SIZE": DEV_SIZE,
            "HOP_EXPANSION": HOP_EXPANSION,
            "DIM": DIM,
            "DEPTH": DEPTH,
            "N1": N1,
            "N2": N2,
            "NUM_ENT": NUM_ENT,
        },
    }, save_path)
    print("Saved", save_path)
except Exception as e:
    alt = f"/content/tea_gnn_tkgl_{LANG_PAIR}_sampled.pt"
    torch.save({"model": model.state_dict()}, alt)
    print("Drive save failed, saved to", alt, ":", e)


embedding shape (11026, 64)
Final CSLS on dev:
  Hits@1: 0.07%  Hits@5: 0.27%  Hits@10: 0.43%  MRR: 0.0022  (n=3000)
Saved /content/drive/MyDrive/tea_gnn_tkgl_en_ja_sampled.pt


## Why your first run got ~0% Hits@1

With `HOP_EXPANSION=0`, only alignment entities were kept. In TKGL almost all facts link an alignment entity to a **non-alignment** neighbor, so the filtered graph had **nearly zero edges**. The GNN then had no structure to learn → random embeddings → Hits@1 ≈ 0.

**This version defaults to `HOP_EXPANSION=1`** (seeds + 1-hop neighbors).

## If you still OOM

1. **Runtime → Restart session** (mandatory after OOM).
2. Reduce sampling:
   ```python
   TRAIN_SEED = 1000
   DEV_SIZE = 1500
   MAX_QUADS_EN = 300_000
   MAX_QUADS_OTHER = 150_000
   ```
3. Smaller model:
   ```python
   DIM = 32
   DEPTH = 1
   NEG_SIZE = 16
   BATCH_PAIRS = 256
   ```
4. Do **not** set `HOP_EXPANSION = 0` unless you only want a smoke test.

## Notes for thesis
- Results are on a **sampled alignment-centric subgraph** (seeds + 1-hop), not full TKGL.
- Document: seed size, hop expansion, max quads, DIM, DEPTH, avg degree.
- Switch language: set `LANG_PAIR` to `en_fa` / `en_zh` / `en_ja` / `en_fr` and re-run from Config.
